In [5]:
from audioop import avg

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

from test.cnn import criterion, outputs

from handwritten_digit_recognition import predict, accuracy_count


In [7]:
print("正在生成序列数据...")
num_samples = 1000
seq_length =10
input_features = 1

X = torch.randn(num_samples,seq_length,input_features)
Y = (X.sum(dim=1).squeeze() > 0).long()

dataset = TensorDataset(X,Y)
train_loader = DataLoader(dataset,batch_size=32,shuffle=True)

正在生成序列数据...


In [8]:
class SimpleRNN(nn.Module):
    def __init__(self):
        super(SimpleRNN,self).__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=16, num_layers=1, batch_first=True)
        self.fc = nn.Linear(in_features=16, out_features=2)

    def forward(self,x):
        out,hideen = self.rnn(x)
        final_memory = out[:,-1,:]

        result = self.fc(final_memory)
        return result

In [9]:
model = SimpleRNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

In [11]:
epochs = 10
print("开始训练 RNN 模型...")
for epoch in range(epochs):
    running_loss = 0.0
    correct_answer = 0
    total_question = 0

    for sequense,labels in train_loader:
        outputs = model(sequense)
        loss = criterion(outputs,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _,predicted = torch.max(outputs,1)
        total_question += labels.size(0)
        correct_answer += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100*correct_answer/total_question
    print(f"第 {epoch+1:2d} 遍学习 | 平均误差(Loss): {avg_loss:.4f} | 准确率: {accuracy:.2f}%")

print("RNN 训练彻底结束！")

开始训练 RNN 模型...
第  1 遍学习 | 平均误差(Loss): 0.1010 | 准确率: 96.00%
第  2 遍学习 | 平均误差(Loss): 0.1541 | 准确率: 93.20%
第  3 遍学习 | 平均误差(Loss): 0.2274 | 准确率: 90.50%
第  4 遍学习 | 平均误差(Loss): 0.1444 | 准确率: 93.80%
第  5 遍学习 | 平均误差(Loss): 0.1074 | 准确率: 95.90%
第  6 遍学习 | 平均误差(Loss): 0.1417 | 准确率: 93.30%
第  7 遍学习 | 平均误差(Loss): 0.1063 | 准确率: 95.10%
第  8 遍学习 | 平均误差(Loss): 0.0973 | 准确率: 96.00%
第  9 遍学习 | 平均误差(Loss): 0.0917 | 准确率: 96.30%
第 10 遍学习 | 平均误差(Loss): 0.1123 | 准确率: 94.60%
RNN 训练彻底结束！
